In [177]:
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# =====================================================
# LOAD DATASET
# =====================================================

sms_df = pd.read_csv(
    "data/sms_spam",
    sep="\t",
    names=["label", "message"]
)

print("=" * 50)
print("FIRST 5 RECORDS")
print("=" * 50)
print(sms_df.head())

print("\n")

print("=" * 50)
print("DATASET INFORMATION")
print("=" * 50)
print(sms_df.info())

print("\n")

print("=" * 50)
print("DATASET SHAPE")
print("=" * 50)
print(sms_df.shape)

print("\n")

print("=" * 50)
print("LABEL DISTRIBUTION")
print("=" * 50)
print(sms_df["label"].value_counts())

print("\n")

print("=" * 50)
print("MISSING VALUES")
print("=" * 50)
print(sms_df.isnull().sum())

# =====================================================
# LABEL ENCODING
# =====================================================

sms_df["label"] = sms_df["label"].map({
    "ham": 0,
    "spam": 1
})

# =====================================================
# FEATURES & TARGET
# =====================================================

X = sms_df["message"]
y = sms_df["label"]

# =====================================================
# TF-IDF
# =====================================================

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

X_tfidf = tfidf.fit_transform(X)

print("\n")
print("=" * 50)
print("TF-IDF SHAPE")
print("=" * 50)
print(X_tfidf.shape)

# =====================================================
# TRAIN TEST SPLIT
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =====================================================
# MODEL TRAINING
# =====================================================

model = MultinomialNB()

model.fit(X_train, y_train)

print("\n")
print("=" * 50)
print("MODEL TRAINED SUCCESSFULLY")
print("=" * 50)

# =====================================================
# PREDICTION
# =====================================================

y_pred = model.predict(X_test)

# =====================================================
# EVALUATION
# =====================================================

accuracy = accuracy_score(y_test, y_pred)

print("\n")
print("=" * 50)
print("ACCURACY")
print("=" * 50)
print(accuracy)

print("\n")
print("=" * 50)
print("CLASSIFICATION REPORT")
print("=" * 50)
print(classification_report(y_test, y_pred))

print("\n")
print("=" * 50)
print("CONFUSION MATRIX")
print("=" * 50)
print(confusion_matrix(y_test, y_pred))

# =====================================================
# TEST CUSTOM SMS
# =====================================================

print("\n")
print("=" * 50)
print("CUSTOM SMS TEST")
print("=" * 50)

sample_sms = [
    "Congratulations! You have won RM5000. Click here now.",
    "Hi, are we still meeting tonight?",
    "URGENT! Your account has been suspended.",
    "Can you send me the lecture notes?"
]

sample_vector = tfidf.transform(sample_sms)

predictions = model.predict(sample_vector)

probabilities = model.predict_proba(sample_vector)

for sms, pred, prob in zip(sample_sms, predictions, probabilities):

    label = "SPAM" if pred == 1 else "HAM"

    print("=" * 50)
    print("SMS:", sms)
    print("Prediction:", label)

    print("Ham Probability :", round(prob[0] * 100, 2), "%")
    print("Spam Probability:", round(prob[1] * 100, 2), "%")
# =====================================================
# SAVE MODEL
# =====================================================

pickle.dump(model, open("sms_model.pkl", "wb"))
pickle.dump(tfidf, open("sms_vectorizer.pkl", "wb"))


print("\n")
print("=" * 50)
print("MODEL SAVED")
print("=" * 50)
print("sms_model.pkl")
print("sms_vectorizer.pkl")

FIRST 5 RECORDS
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


DATASET INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5572 non-null   object
 1   message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB
None


DATASET SHAPE
(5572, 2)


LABEL DISTRIBUTION
label
ham     4825
spam     747
Name: count, dtype: int64


MISSING VALUES
label      0
message    0
dtype: int64


TF-IDF SHAPE
(5572, 5000)


MODEL TRAINED SUCCESSFULLY


ACCURACY
0.9730941704035875


CLASSIFICATION REPORT
              precision    recall  f

URL

In [178]:
import pandas as pd
import numpy as np
import pickle
import re
import math

from urllib.parse import urlparse
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, classification_report

In [179]:
url_df = pd.read_csv("data/malicious_phish.csv")

X = url_df["url"]
y = url_df["type"]

print(url_df.head())
print(url_df["type"].value_counts())

                                                 url        type
0                                   br-icloud.com.br    phishing
1                mp3raid.com/music/krizz_kaliko.html      benign
2                    bopsecrets.org/rexroth/cr/1.htm      benign
3  http://www.garage-pirenne.be/index.php?option=...  defacement
4  http://adventure-nicaragua.net/index.php?optio...  defacement
type
benign        428103
defacement     96457
phishing       94111
malware        32520
Name: count, dtype: int64


In [180]:
tfidf = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    max_features=10000
)

X_tfidf = tfidf.fit_transform(X)

In [181]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [182]:
base_model = LinearSVC()

model = CalibratedClassifierCV(
    base_model,
    cv=3
)
model.fit(X_train, y_train)

,estimator,LinearSVC()
,method,'sigmoid'
,cv,3
,n_jobs,None
,ensemble,'auto'
,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'


In [183]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9696864994356529
              precision    recall  f1-score   support

      benign       0.97      0.99      0.98     85621
  defacement       0.99      1.00      0.99     19292
     malware       0.98      0.94      0.96      6504
    phishing       0.92      0.88      0.90     18822

    accuracy                           0.97    130239
   macro avg       0.97      0.95      0.96    130239
weighted avg       0.97      0.97      0.97    130239



In [184]:
TRUSTED_DOMAINS = {
    "google.com",
    "facebook.com",
    "amazon.com",
    "microsoft.com",
    "apple.com",
    "paypal.com",
    "github.com",
    "wikipedia.org"
}

SUSPICIOUS_TLDS = {
    ".xyz", ".top", ".click", ".tk", ".ml", ".ga", ".cf"
}

SHORTENERS = {
    "bit.ly", "tinyurl.com", "t.co"
}

SUSPICIOUS_KEYWORDS = {
    "login", "verify", "secure", "update", "signin", "account"
}

KNOWN_BRANDS = {
    "google", "facebook", "amazon", "paypal", "microsoft", "apple"
}

CRITICAL_FLAGS = {"ip_url", "brand_spoofing"}

def extract_domain(url):
    parsed = urlparse(url if url.startswith("http") else "http://" + url)
    return parsed.netloc.lower()


def trust_engine(url):
    domain = extract_domain(url)

    if domain in TRUSTED_DOMAINS:
        return 20  # keep small, NOT overpower ML

    # subdomain of trusted (safe-ish)
    for trusted in TRUSTED_DOMAINS:
        if domain.endswith("." + trusted):
            return 10

    return 0

def rule_engine(url):
    score = 0
    flags = []

    url_lower = url.lower()
    parsed = urlparse(url if url.startswith("http") else "http://" + url)

    domain = parsed.netloc.lower()
    path = parsed.path.lower()

    # ---------------- IP ----------------
    if re.search(r"\d+\.\d+\.\d+\.\d+", url_lower):
        score += 25
        flags.append("ip_url")

    # ---------------- HTTPS ----------------
    if url_lower.startswith("http://"):
        score += 8
        flags.append("no_https")

    # ---------------- TLD ----------------
    if any(domain.endswith(tld) for tld in SUSPICIOUS_TLDS):
        score += 20
        flags.append("suspicious_tld")

    # ---------------- SHORTENER ----------------
    if any(s in domain for s in SHORTENERS):
        score += 25
        flags.append("shortener")

    # ---------------- KEYWORDS (PATH ONLY) ----------------
    if any(k in path for k in SUSPICIOUS_KEYWORDS):
        score += 15
        flags.append("suspicious_keywords")

    # ---------------- LONG URL ----------------
    if len(url) > 100:
        score += 10
        flags.append("long_url")

    # ---------------- BRAND SPOOFING (FIXED) ----------------
    for brand in KNOWN_BRANDS:
        if brand in domain:
            official = f"{brand}.com"

            if domain != official and not domain.endswith("." + official):
                score += 35
                flags.append("brand_spoofing")
                break

    return min(score, 100), flags

def ml_engine(url):
    vec = tfidf.transform([url])
    prob = model.predict_proba(vec)[0]

    phishing_index = list(model.classes_).index("phishing")
    phishing_prob = prob[phishing_index]

    return phishing_prob * 100

def fusion_engine(url):

    rule_score, rule_flags = rule_engine(url)
    trust_score = trust_engine(url)
    ml_score = ml_engine(url)


    if any(flag in CRITICAL_FLAGS for flag in rule_flags):
        return {
            "url": url,
            "final_score": 90,
            "label": "HIGH_RISK",
            "rule_score": rule_score,
            "ml_score": ml_score,
            "trust_score": trust_score,
            "flags": rule_flags
        }

    # ---------------- FUSION ----------------
    final_score = (
        rule_score * 0.5 +
        ml_score * 0.4 -
        trust_score * 0.3
    )

    final_score = max(0, min(100, final_score))

    if final_score >= 70:
        label = "HIGH_RISK"
    elif final_score >= 40:
        label = "SUSPICIOUS"
    else:
        label = "SAFE"

    return {
        "url": url,
        "final_score": round(final_score, 2),
        "label": label,
        "rule_score": rule_score,
        "ml_score": round(ml_score, 2),
        "trust_score": trust_score,
        "flags": rule_flags
    }

In [185]:
test_urls = [
    "google.com",
    "facebook.com",
    "amazon.com",
    "paypal.com",

    "google-login.xyz",
    "amazon-secure-update.top",
    "paypal-login-security-update.com",

    "http://192.168.1.1/login",
    "http://bit.ly/login-secure-update",

    "https://accounts.google.com"
]

for url in test_urls:

    result = fusion_engine(url)

    print("="*50)
    print("URL:", result["url"])
    print("LABEL:", result["label"])
    print("FINAL SCORE:", result["final_score"])
    print("RULE SCORE:", result["rule_score"])
    print("ML SCORE:", result["ml_score"])
    print("TRUST SCORE:", result["trust_score"])
    print("FLAGS:", result["flags"])

URL: google.com
LABEL: SAFE
FINAL SCORE: 33.44
RULE SCORE: 0
ML SCORE: 98.6
TRUST SCORE: 20
FLAGS: []
URL: facebook.com
LABEL: SAFE
FINAL SCORE: 26.4
RULE SCORE: 0
ML SCORE: 81.0
TRUST SCORE: 20
FLAGS: []
URL: amazon.com
LABEL: SAFE
FINAL SCORE: 33.82
RULE SCORE: 0
ML SCORE: 99.56
TRUST SCORE: 20
FLAGS: []
URL: paypal.com
LABEL: SAFE
FINAL SCORE: 33.3
RULE SCORE: 0
ML SCORE: 98.25
TRUST SCORE: 20
FLAGS: []
URL: google-login.xyz
LABEL: HIGH_RISK
FINAL SCORE: 90
RULE SCORE: 55
ML SCORE: 92.3780446626263
TRUST SCORE: 0
FLAGS: ['suspicious_tld', 'brand_spoofing']
URL: amazon-secure-update.top
LABEL: HIGH_RISK
FINAL SCORE: 90
RULE SCORE: 55
ML SCORE: 50.85283755312409
TRUST SCORE: 0
FLAGS: ['suspicious_tld', 'brand_spoofing']
URL: paypal-login-security-update.com
LABEL: HIGH_RISK
FINAL SCORE: 90
RULE SCORE: 35
ML SCORE: 76.5008379881882
TRUST SCORE: 0
FLAGS: ['brand_spoofing']
URL: http://192.168.1.1/login
LABEL: HIGH_RISK
FINAL SCORE: 90
RULE SCORE: 48
ML SCORE: 4.645927000730007
TRUST SCO

In [186]:
import inspect

print(inspect.getsource(fusion_engine))

def fusion_engine(url):

    rule_score, rule_flags = rule_engine(url)
    trust_score = trust_engine(url)
    ml_score = ml_engine(url)


    if any(flag in CRITICAL_FLAGS for flag in rule_flags):
        return {
            "url": url,
            "final_score": 90,
            "label": "HIGH_RISK",
            "rule_score": rule_score,
            "ml_score": ml_score,
            "trust_score": trust_score,
            "flags": rule_flags
        }

    # ---------------- FUSION ----------------
    final_score = (
        rule_score * 0.5 +
        ml_score * 0.4 -
        trust_score * 0.3
    )

    final_score = max(0, min(100, final_score))

    if final_score >= 70:
        label = "HIGH_RISK"
    elif final_score >= 40:
        label = "SUSPICIOUS"
    else:
        label = "SAFE"

    return {
        "url": url,
        "final_score": round(final_score, 2),
        "label": label,
        "rule_score": rule_score,
        "ml_score": round(ml_score, 2),
        "trust_sco